### TOOLS

In [ ]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=ChatGroq(model="qwen/qwen3.6-27b", reasoning_effort="none")
response = model.invoke("Why do parrots talk?")
response

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"it's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [ ]:
response = model_with_tools.invoke("what is the weather like in Boston")
print(response)
for tool_call in response.tool_calls:
    print(f"tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

### Tool Execution Loops

In [7]:
# step 1: Model generates tool calls
messages = [
    {
        "role": "user",
        "content": "What's the weather in Boston?"
    }
]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tools with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72 degree F and sunny."

It's sunny in Boston.


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hssxx80x9', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 277, 'total_tokens': 303, 'completion_time': 0.049581623, 'completion_tokens_details': None, 'prompt_time': 0.021069043, 'prompt_tokens_details': None, 'queue_time': 0.046850207, 'total_time': 0.070650666}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_49d6b1859d', 'service_tier': 'on_demand', 'reasoning_effort': 'none', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a02442-a5bf-7c92-9722-939659282626-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'hssxx80x9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 277, 'output_tokens': 26, 'total_tokens': 303}),
 Tool